# Ćwiczenia 2 — Eksploracja danych przed modelowaniem (wersja Python)
**Dane demo: kohorta T2D (QinJ_2012)**

Model uczy się wyłącznie z tego, co dostanie w danych. Jeśli dane zawierają błędy, przypadkowe zależności albo cechy, które mierzą coś innego niż badane zjawisko, model nauczy się także tego. Dlatego przed budową pierwszego modelu sprawdzamy: jakość danych, czy pojedyncze cechy w ogóle różnią klasy, czy różnic nie tłumaczy inna zmienna, które cechy powtarzają tę samą informację i jak skala cech wpływa na podobieństwo próbek.

Na tym spotkaniu niczego nie usuwamy ani nie przekształcamy. Wyniki służą do zaplanowania decyzji, które podejmuje się dopiero po podziale danych na zbiór treningowy i testowy: decyzja podjęta na całym zbiorze korzystałaby z informacji o próbkach testowych.

## Wczytanie danych

Pliki CSV z katalogu `dane/` wyeksportowano na Ćwiczeniach 1. Są trzy, bo każdy przechowuje co innego: `X` to cechy (wiersze to próbki, kolumny to taksony, wartości to liczebności względne w procentach), `y` to klasa każdej próbki, a `meta` to metadane próbek (np. wiek). Notatnik najlepiej uruchamiać od początku po ponownym uruchomieniu jądra (*Kernel → Restart*), komórka po komórce: obiekty pozostawione w pamięci przez wcześniejszy kod mogą mieć te same nazwy i inną zawartość.

In [ ]:
import numpy as np                       # obliczenia na tablicach
import pandas as pd                      # tabele danych
import matplotlib.pyplot as plt          # wykresy
pd.set_option("display.width", 120)      # szersze wydruki tabel (bez zawijania kolumn)
pd.set_option("display.max_columns", 20) # wszystkie kolumny tabeli na wydruku

# index_col=0: pierwsza kolumna pliku to nazwy próbek, a nie cecha
X = pd.read_csv("dane/t2d_features.csv", index_col=0)
# ["grupa"]: kolumna z etykietami; .to_numpy(): zwykła tablica napisów "control"/"T2D"
y = pd.read_csv("dane/t2d_labels.csv")["grupa"].to_numpy()
meta = pd.read_csv("dane/t2d_metadata.csv", index_col=0)   # metadane; nazwy próbek jako indeks

print(X.shape)                           # (liczba próbek, liczba cech)
print(pd.Series(y).value_counts())       # liczba próbek w każdej klasie
print(list(meta.columns))                # nazwy kolumn metadanych

## 1. Jakość danych i nietypowe próbki

Błędy w danych przechodzą do modelu. Brak danych zatrzymuje wiele funkcji albo jest pomijany bez ostrzeżenia. Powtórzona próbka może trafić jednocześnie do zbioru treningowego i testowego, a wtedy ocena modelu jest zawyżona. Etykieta przypisana do niewłaściwej próbki sprawia, że model uczy się szumu. Plik etykiet nie zawiera nazw próbek, więc `y` odpowiada wierszom `X` tylko według kolejności; porównanie z etykietą zapisaną w metadanych sprawdza, czy kolejność się zgadza.

In [ ]:
print(X.isna().sum().sum())              # .isna() daje True przy braku; dwa .sum(): po kolumnach, potem razem
print(meta.isna().sum())                 # liczba braków osobno w każdej kolumnie metadanych
print(X.duplicated().sum())              # .duplicated(): True dla wiersza, który wystąpił już wcześniej
print((meta["study_condition"].to_numpy() == y).all())   # .all(): True, gdy etykiety zgadzają się wszędzie
# część nazwy przed dwukropkiem to poziom taksonomiczny, np. "species"
print(X.columns.str.split(":").str[0].value_counts())

**Na co patrzeć.** W danych demo nie ma braków ani duplikatów, a etykiety odpowiadają próbkom. Cechy nie są jednorodne: cztery opisują rodzaj (*genus*), a pozostałe gatunek. Cecha na poziomie rodzaju może częściowo obejmować te same organizmy co cechy gatunkowe.

Próbka nietypowa może sama przesuwać średnie, odległości i wyniki modelu. Trzeba ją znaleźć i rozstrzygnąć, czy to błąd (wtedy się ją usuwa), czy rzadki, ale prawdziwy przypadek (wtedy zostaje, bo model ma działać także na takich próbkach). Dla każdej próbki liczymy dwa podsumowania profilu: liczbę cech o wartości niezerowej i udział największej cechy. Próbkę oznaczamy, gdy podsumowanie leży dalej niż 1,5 rozstępu międzykwartylowego (IQR) od pierwszego lub trzeciego kwartyla (reguła z wykładu 2).

In [ ]:
n_nonzero = (X > 0).sum(axis=1)          # X > 0 daje True/False; .sum(axis=1) liczy True w każdym wierszu
max_share = X.max(axis=1)                # największa wartość w każdym wierszu (próbce)

# własna funkcja: dla serii v zwraca True przy wartościach odstających według reguły k × IQR
def iqr_outlier(v, k=1.5):
    q1, q3 = np.quantile(v, [0.25, 0.75])                        # pierwszy i trzeci kwartyl
    return (v < q1 - k * (q3 - q1)) | (v > q3 + k * (q3 - q1))   # q3 − q1 to IQR; | oznacza „lub”

# tabela z dwiema kolumnami True/False, po jednym wierszu na próbkę
flags = pd.DataFrame({"n_nonzero": iqr_outlier(n_nonzero),
                      "max_share": iqr_outlier(max_share)})
print(flags.sum())                       # ile próbek oznacza każda reguła
outlier = flags.any(axis=1)              # True, gdy próbkę oznaczyła przynajmniej jedna reguła
print(pd.crosstab(outlier.rename("nietypowa"), y))   # tabela krzyżowa: oznaczenie × klasa

# wykres: jedna próbka = jeden punkt; kolor pokazuje, czy próbkę oznaczono (~ oznacza „nie”)
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(n_nonzero[~outlier], max_share[~outlier], facecolors="none", edgecolors="grey", label="nie")
ax.scatter(n_nonzero[outlier], max_share[outlier], facecolors="none", edgecolors="#B2182B", label="tak")
ax.set_xlabel("Liczba cech niezerowych w próbce")
ax.set_ylabel("Udział największej cechy (%)")
ax.legend(title="nietypowa")
plt.show()

Które cechy dominują w próbkach oznaczonych przez drugą regułę:

In [ ]:
dominant = X.idxmax(axis=1).str.replace("species:", "")   # .idxmax(axis=1): nazwa kolumny z największą wartością
flagged = pd.DataFrame({"max_share": max_share.round(1),   # udział największej cechy
                        "dominant": dominant,              # jej nazwa
                        "group": y})[flags["max_share"]]   # tylko wiersze oznaczone przez max_share
print(flagged.sort_values("max_share", ascending=False))  # od największego udziału

🔧 **Do sprawdzenia.** Wywołać `iqr_outlier()` z `k=3`. Ile próbek zostaje oznaczonych?

> **✏️ Zadanie 1.** Na podstawie tabeli `flagged`: która cecha dominuje w oznaczonych próbkach i w której klasie jest ich więcej? Czy próbki te wyglądają na błędy pomiaru, czy na rzadszy, ale prawdziwy typ profilu? Czy należy je usunąć przed modelowaniem? Odpowiedzieć w 2–3 zdaniach pod poleceniem.

## 2. Pojedyncze cechy a klasa

Zanim zbudujemy model, sprawdzamy, czy klasy w ogóle różnią się cechami, którymi się różnią i w jaki sposób. Jeśli żadna cecha nie różni klas, trudno oczekiwać dobrego modelu; jeśli różni ich wiele, wiadomo, gdzie szukać wyjaśnienia jego przewidywań. To przegląd danych, a nie wybór cech do modelu: na tej podstawie niczego nie usuwamy. Wybór cech według etykiet wykonany na całym zbiorze korzysta z próbek testowych i zawyża ocenę modelu, dlatego wybór cech, jeśli jest potrzebny, wykonuje się później wyłącznie na zbiorze treningowym.

W danych z wieloma zerami cecha może różnić klasy na dwa sposoby: tym, jak często występuje, albo tym, ile jej jest, gdy występuje. Dlatego dla każdej cechy liczymy dwa podsumowania w każdej klasie: częstość występowania (odsetek próbek, w których cecha jest większa od zera) i średnią. Różnicę średnich podajemy jako log2 ilorazu średnich (T2D / control): 1 oznacza średnią dwa razy większą w T2D, −1 dwa razy mniejszą, 0 brak różnicy. Bierzemy tylko cechy obecne w co najmniej 10% wszystkich próbek, bo dla cechy prawie zawsze równej zeru oba podsumowania opierają się na kilku próbkach.

In [ ]:
prevalence = (X > 0).mean(axis=0)        # częstość występowania w całym zbiorze
Xf = X.loc[:, prevalence >= 0.10]        # zostają kolumny (cechy) obecne w co najmniej 10% próbek
print(Xf.shape[1])                       # liczba zachowanych cech

is_t2d = y == "T2D"                      # True dla próbek T2D, False dla control
prev_control = (Xf[~is_t2d] > 0).mean()  # częstość występowania w control (~ oznacza „nie”)
prev_t2d = (Xf[is_t2d] > 0).mean()       # częstość występowania w T2D
diff_prev = prev_t2d - prev_control      # dodatnia: cecha występuje częściej w T2D

mean_control = Xf[~is_t2d].mean()        # średnia cechy w control
mean_t2d = Xf[is_t2d].mean()             # średnia cechy w T2D
# log2 ilorazu średnich; + 1e-4 (najmniejsza niezerowa wartość w danych) chroni przed dzieleniem przez 0
log2_ratio = np.log2((mean_t2d + 1e-4) / (mean_control + 1e-4))

# 10 cech o największej różnicy częstości występowania (abs(): bez względu na kierunek)
top = diff_prev.abs().sort_values(ascending=False).index[:10]
table = pd.DataFrame({"prev_control": prev_control[top], "prev_t2d": prev_t2d[top],
                      "diff_prev": diff_prev[top], "log2_ratio": log2_ratio[top]}).round(2)
table.index = table.index.str.replace("species:", "")   # krótsze nazwy wierszy
print(table)

# wykres: jedna cecha = jeden punkt; częstość w control na osi X, w T2D na osi Y
in_top = prev_control.index.isin(top)    # True dla 10 cech z tabeli
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], linestyle="--", color="black")   # przekątna: ta sama częstość w obu klasach
ax.scatter(prev_control[~in_top], prev_t2d[~in_top], facecolors="none", edgecolors="grey", label="pozostałe")
ax.scatter(prev_control[in_top], prev_t2d[in_top], facecolors="none", edgecolors="#B2182B", label="10 z tabeli")
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")   # jednakowa skala obu osi
ax.set_xlabel("Częstość występowania w control"); ax.set_ylabel("Częstość występowania w T2D")
ax.legend()
plt.show()

**Na co patrzeć.** Punkty na przekątnej to cechy występujące równie często w obu klasach. Punkty nad przekątną występują częściej w T2D, pod nią częściej w control. Większość cech leży blisko przekątnej, więc klasy różnią się tylko częścią cech i w niewielkim stopniu.

Rozkład pięciu cech z początku tabeli w obu klasach. Używamy skali log10, bo wartości obejmują kilka rzędów wielkości; do zer dodajemy `1e-6`, bo logarytm z zera nie istnieje, więc zera leżą przy −6.

In [ ]:
show = list(top[:5])                     # cechy do pokazania; można tu wpisać inne nazwy, np. ["genus:Streptococcus"]
fig, axes = plt.subplots(1, len(show), figsize=(2.8 * len(show), 3.5), sharey=True, squeeze=False)   # panele w rzędzie
for ax, feature in zip(axes[0], show):   # jeden panel na jedną cechę
    values = np.log10(Xf[feature] + 1e-6)                       # wartości cechy w skali log10
    ax.boxplot([values[~is_t2d], values[is_t2d]], tick_labels=["control", "T2D"])   # pudełko na klasę
    ax.set_title(feature.replace("species:", ""), fontsize=9)  # nazwa cechy nad panelem
axes[0][0].set_ylabel("log10(liczebność względna + 1e-6)")
plt.tight_layout()                       # dopasowanie odstępów między panelami
plt.show()

**Na co patrzeć.** Gdy cechy nie ma w większości próbek, mediana i dolny brzeg pudełka leżą przy −6. Różnica między klasami polega wtedy głównie na tym, jak często cecha występuje, a nie na jej typowej wartości.

> **✏️ Zadanie 2.** W tabeli znaleźć cechę, dla której `diff_prev` i `log2_ratio` mają przeciwne znaki. W pierwszym wierszu bloku z wykresami pudełkowymi wpisać jej nazwę (`show = ["..."]`) i narysować wykres. Wypisać jej pięć największych wartości w każdej klasie (`Xf.loc[~is_t2d, "..."].nlargest(5)` i to samo dla T2D). Skąd bierze się niezgodność obu podsumowań i któremu z nich bardziej ufać?

## 3. Zmienna zakłócająca

Na Ćwiczeniach 1 okazało się, że klasy różnią się wiekiem: grupa T2D jest starsza. Cecha może więc różnić klasy dlatego, że jest związana z wiekiem, a nie z chorobą. Model, który ją wykorzysta, rozpozna wtedy wiek, a nie T2D, i zawiedzie na danych, w których grupy mają podobny wiek. Taką zmienną nazywa się zakłócającą (ang. *confounder*). Sprawdzamy to po to, żeby wiedzieć, jak ostrożnie interpretować cechy, na których oprze się model. Blok pokazuje wiek w obu klasach i liczy korelację Spearmana (opartą na rangach, bo rozkłady cech są skośne) każdej z 10 cech z tabeli z wiekiem.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.5))
ax.boxplot([meta["age"][~is_t2d], meta["age"][is_t2d]], tick_labels=["control", "T2D"])   # wiek w każdej klasie
ax.set_ylabel("Wiek (lata)")
plt.show()

# korelacja Spearmana każdej cechy z top z wiekiem; pętla w nawiasach liczy ją po kolei dla każdej cechy
rho_age = pd.Series([Xf[f].corr(meta["age"], method="spearman") for f in top], index=top)
age_table = pd.DataFrame({"diff_prev": diff_prev[top],   # kierunek różnicy między klasami (z sekcji 2)
                          "rho_age": rho_age}).round(2)  # korelacja z wiekiem: od −1 do 1, 0 = brak związku
age_table.index = age_table.index.str.replace("species:", "")   # krótsze nazwy wierszy
print(age_table)

> **✏️ Zadanie 3.** Na podstawie tabeli: czy cechy częstsze w T2D (dodatni `diff_prev`) rosną, czy maleją z wiekiem, a jak jest z cechami rzadszymi w T2D? Czy można twierdzić, że cechy te różnią klasy z powodu choroby? Odpowiedzieć w 2–3 zdaniach.

## 4. Cechy powtarzające tę samą informację

Dwie silnie skorelowane cechy niosą prawie tę samą informację. Model nie zyskuje na drugiej z nich, a niektóre modele nie potrafią rozdzielić wpływu między takie cechy, więc wynik zależy od przypadku, a wkładu każdej z cech nie da się ocenić osobno. Sprawdzamy, ile jest takich par, żeby później zdecydować: zostawić jedną cechę z pary, połączyć cechy albo użyć modelu, który radzi sobie z korelacją. Korelację liczymy metodą Spearmana (na rangach), bo rozkłady cech są skośne.

In [ ]:
r = Xf.corr(method="spearman")                         # macierz korelacji: wiersz i kolumna dla każdej cechy
r_pairs = r.to_numpy()[np.triu_indices_from(r, k=1)]   # górny trójkąt: każda para raz, bez przekątnej (r = 1)
print(len(r_pairs))                                    # liczba wszystkich par cech
print((np.abs(r_pairs) > 0.9).sum())                   # pary bardzo silnie skorelowane; abs uwzględnia korelacje ujemne
print((np.abs(r_pairs) > 0.7).sum())                   # pary silnie skorelowane

Mapa korelacji dla 20 cech o największej średniej. Kolejność cech ustala grupowanie hierarchiczne, które ustawia podobne cechy obok siebie, dzięki czemu grupy skorelowanych cech widać jako bloki.

In [ ]:
from scipy.cluster.hierarchy import leaves_list, linkage   # grupowanie hierarchiczne
from scipy.spatial.distance import squareform              # zamiana macierzy odległości na postać dla linkage()

top20 = Xf.mean().sort_values(ascending=False).index[:20]  # 20 cech o największej średniej
r20 = r.loc[top20, top20]                                  # fragment macierzy korelacji dla tych cech
# odległość 1 − r; leaves_list() zwraca kolejność cech po grupowaniu
order = leaves_list(linkage(squareform(1 - r20.to_numpy(), checks=False), method="complete"))
r20 = r20.iloc[order, order]                               # wiersze i kolumny w nowej kolejności
names = [n.replace("species:", "") for n in r20.index]     # krótsze nazwy do podpisów

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(r20, cmap="RdBu_r", vmin=-1, vmax=1)        # mapa: niebieski < 0 < czerwony
ax.set_xticks(range(20), names, rotation=90, fontsize=8)  # pionowe podpisy osi X
ax.set_yticks(range(20), names, fontsize=8)
fig.colorbar(im, label="korelacja Spearmana")             # skala kolorów
plt.tight_layout()
plt.show()

**Na co patrzeć.** Bloki czerwonych komórek to grupy cech, które rosną i maleją razem. W danych z dużą liczbą zer korelację podnoszą też wspólne zera: dwie cechy nieobecne w tych samych próbkach mają wysoką korelację rang.

Para cech o największej korelacji i jej wykres rozrzutu:

In [ ]:
r0 = r.to_numpy().copy()                               # kopia macierzy (tablica z .corr() jest tylko do odczytu)
np.fill_diagonal(r0, 0)                                # pomijamy korelację cechy z samą sobą
i, j = np.unravel_index(np.abs(r0).argmax(), r0.shape) # wiersz i kolumna największej |r|
a, b = Xf.columns[i], Xf.columns[j]                    # nazwy obu cech pary
print(a, "|", b)                                       # wypisanie nazw
print(round(r0[i, j], 3))                              # ich korelacja

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(Xf[a], Xf[b], facecolors="none", edgecolors="grey")
ax.set_xlabel(a); ax.set_ylabel(b)
plt.show()

**Na co patrzeć.** Obie cechy mają tę samą nazwę gatunku: w danych ten sam gatunek występuje dwa razy, raz z końcówką `_1` w nazwie, raz bez niej. Punkty leżą prawie na prostej, więc druga cecha nie wnosi nowej informacji. Taką parę należy wykryć przed modelowaniem i zostawić jedną z cech.

🔧 **Do sprawdzenia.** Ile par ma |r| > 0,8? Zmienić próg w jednym z wierszy bloku z liczeniem par.

## 5. Skala cech i podobieństwo próbek

Metody nienadzorowane (np. analiza głównych składowych i grupowanie, omawiane na następnych ćwiczeniach) oraz część modeli oceniają, które próbki są do siebie podobne, zwykle za pomocą odległości euklidesowej. Odległość sumuje różnice we wszystkich cechach, więc o wyniku decydują cechy o największym rozrzucie. Sprawdzamy, jak rozkłada się wariancja między cechami i jak bardzo odległości między próbkami zależą od jednej cechy, żeby wiedzieć, czy przed takimi metodami trzeba dane przekształcić.

In [ ]:
from scipy.spatial.distance import pdist                  # odległości między wszystkimi parami wierszy

variances = Xf.var().sort_values(ascending=False)          # wariancja każdej cechy, od największej
print((variances / variances.sum()).head(5).round(3))      # udział 5 pierwszych cech w całkowitej wariancji
print(round(variances.iloc[:5].sum() / variances.sum(), 3))   # łączny udział tych 5 cech

# wykres: średnia i odchylenie standardowe każdej cechy
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(Xf.mean(), Xf.std(), facecolors="none", edgecolors="black")
ax.set_xscale("log"); ax.set_yscale("log")                 # osie logarytmiczne: wartości z kilku rzędów wielkości
ax.set_xlabel("Średnia cechy (%)"); ax.set_ylabel("Odchylenie standardowe cechy")
plt.show()

d_top1 = pdist(Xf[[variances.index[0]]])                   # odległości między próbkami liczone tylko na jednej cesze
d_raw = pdist(Xf)                                          # odległości euklidesowe liczone na wszystkich cechach
print(np.corrcoef(d_raw, d_top1)[0, 1])                    # na ile ta jedna cecha wyznacza odległości

**Na co patrzeć.** Cechy o większej średniej mają większy rozrzut, więc kilka najliczniejszych cech skupia większość całkowitej wariancji. Im bliżej 1 jest ostatnia liczba, tym bardziej odległości między próbkami wyznacza jedna cecha; przy 190 cechach już korelacja rzędu 0,5 oznacza, że jedna cecha przeważa nad wszystkimi pozostałymi.

> **✏️ Zadanie 4.** W wierszu `d_raw = pdist(Xf)` zamienić `Xf` najpierw na `(Xf - Xf.mean()) / Xf.std()` (standaryzacja: każda cecha ma średnią 0 i odchylenie 1), a potem na `np.log10(Xf + 1e-6)`, i za każdym razem uruchomić dwa ostatnie wiersze bloku. Jak zmienia się korelacja? Co z tego wynika dla metod, które oceniają podobieństwo próbek na podstawie odległości?